In [ ]:
# autores: ai-page.readthedocs.io

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Especificación e identificación

los problemas de *especificación* surgen cuando la insesgadez o consistencia de los parámetros se ve afectada, como en los siguientes casos:

* _Omisión de variables relevantes_ cuando se omiten variables relevantes se puede generar sesgadez de las estimaciones.

* _Inclusión de variables irrelevantes_ esto no sesga pero si afecta la varianza de las estimaciones ( se vuelven ineficientes) y por lo tanto las pruebas de hipótesis.

Nótese que la especificación tiene que ver con las variables mismas del modeo (inclusión o exclusión), en términos generales con la *forma funcional* (logarítmos , exponentes, etc....)  un problema de especificación surgiría cuando tratamos de modelar el crecimiento exponencial con una línea recta.



Algunos test a considerar:
* Test de ramnsey
* Test de hausman
* Visuales


# Identificación

Los problemas de identificación surgen cuando no es posible estimar de manera única (causal?) y precisa los coeficientes, por ejemplo, cuando hay problemas con los datos o en hay problemas en su *especificación*.


véase el problema de ecucaciones simultáneas, la identificación es como estimar de manera única los parámetros.



** Nota:

Es importante destacar la IA hoy aporta metodologías para tatar la especificación usando por ejemplo: regresión simbólica.

In [ ]:
# simular test de ramsey
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import linear_reset

In [ ]:
import statsmodels.api as sm
np.random.seed(123)
beta_1 = 0.4
beta_3 = 1.3
beta_2 = 3.3
n = 100
x1 = np.random.randint(1, 650, size = n)
x2 = np.random.uniform(10, 500, size = n)
x3 = np.random.normal(40,5, size = n)
y = x1*beta_1 + (x2**2)*beta_2  + x3*beta_3 +  np.random.normal(0, 200, size=n) # quadratic term x2

df = pd.DataFrame({'x1':x1, 'x2':x2, 'y':y})


modelo1 = smf.ols('y ~ x1 + x2', data =df).fit()
print(modelo1.summary())
# RESET test: potencia=2 (cuadrática), usamos F-test
reset_test = linear_reset(modelo1, power=2, use_f=True, test_type='fitted')
# la hipotesis nula es que la especificaicón es correcta
p_value = reset_test.pvalue
print(p_value)
# para este caso como p<0.05 entonces no está bien especificado.

modelo2 = smf.ols("y ~ x1 + I(x2**2)", data= df).fit()
print(modelo2.summary())
ram_test = linear_reset(modelo2, power=2, use_f=True, test_type='fitted')
print(ram_test.pvalue)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.943
Model:                            OLS   Adj. R-squared:                  0.941
Method:                 Least Squares   F-statistic:                     796.5
Date:                Thu, 15 May 2025   Prob (F-statistic):           6.39e-61
Time:                        00:07:05   Log-Likelihood:                -1236.1
No. Observations:                 100   AIC:                             2478.
Df Residuals:                      97   BIC:                             2486.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -1.614e+05   1.63e+04     -9.899      0.0

# Omisión de variables relevantes:

Tenemos un modelo poblacional:
$$ y = \beta_{0} + \beta_{1} x_{1} + \beta_{2} x_{2}  + \epsilon $$

Ahora tendremos dos situaciones estimaresmos el modelo:

$$ y = \beta_{0} + \beta_{1} x_{1} + u$$

donde $u = \epsilon  +  \beta_{2} x_{2}$.

y supondremos dos situaciones:

* $cov(x_{1}, x_{2}) \neq 0 $
* $cov(x_{1}, x_{2} ) = 0$

Esto lo hacemos para ver como la omisión de una variable relevante sesga las estimaciones solo si presenta el primer caso (donde las variables explicativas están relacionadas).

In [ ]:
# Primer caso
np.random.seed(1234)
size = 300
x1 = np.random.normal(0,1, size) + np.linspace(1, 100, size)
x2 = np.random.uniform(0,1,size)*x1
b1 = 67.15
b2 = 24.86
b0  = 3
y = b0 + (b1 * x1) + (b2*x2) + np.random.normal(0,150,size)
df = pd.DataFrame({'y':y, 'x1':x1, 'x2':x2})
modelo = smf.ols("y ~ x1 ", data=df).fit()
print(modelo.summary())
# note que el coeficiente b1 está sesgado no es igual a 67.15

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.960
Model:                            OLS   Adj. R-squared:                  0.960
Method:                 Least Squares   F-statistic:                     7092.
Date:                Thu, 15 May 2025   Prob (F-statistic):          8.09e-210
Time:                        00:07:05   Log-Likelihood:                -2265.7
No. Observations:                 300   AIC:                             4535.
Df Residuals:                     298   BIC:                             4543.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     30.2553     54.081      0.559      0.5

In [ ]:
# Segundo caso
np.random.seed(1234)
x1 = np.random.normal(0,1, size) + np.linspace(1, 100, size)
x2 = np.random.uniform(0,1,size)
b1 = 67.15
b2 = 24.86
b0 = 3
y = b0 + b1 * x1 + b2*x2 + np.random.normal(0,150,size)
df = pd.DataFrame({'y':y, 'x1':x1, 'x2':x2})
modelo = smf.ols("y ~ x1 ", data=df).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.994
Model:                            OLS   Adj. R-squared:                  0.994
Method:                 Least Squares   F-statistic:                 4.768e+04
Date:                Thu, 15 May 2025   Prob (F-statistic):               0.00
Time:                        00:07:05   Log-Likelihood:                -1933.3
No. Observations:                 300   AIC:                             3871.
Df Residuals:                     298   BIC:                             3878.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.9138     17.855      1.339      0.1

# Inclusión de variables irrelevanes
En este caso no se sesgan los estimadores pero se la estimación no es eficiente su error estándar crece.

In [ ]:
# Primer caso
np.random.seed(1234)
x1 = np.random.normal(0,1, size) + np.linspace(1, 100, size)
x2 = np.random.uniform(0,1,size) + np.linspace(400, 200, size)
x3 = np.random.normal(0,25, size)
x4 = np.random.normal(100, 3, size) + np.linspace(1200,1900, size)
x5 = np.random.uniform(0, 1, size)
b2 = 24.86
b0 = 3
y = b0 + b1*x1 + b2*x2 + np.random.normal(0,90,size)
df = pd.DataFrame({'y':y, 'x1':x1, 'x2':x2, 'x3':x3})
modelo = smf.ols("y ~ x1 + x2 + x3 + x4 + x5", data=df).fit()
print(modelo.summary())
modelo = smf.ols("y ~ x1 + x2", data=df).fit()
print(modelo.summary())
# note como incremento el std error de x1.

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.972
Model:                            OLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                     2058.
Date:                Thu, 15 May 2025   Prob (F-statistic):          2.16e-226
Time:                        00:09:42   Log-Likelihood:                -1755.4
No. Observations:                 300   AIC:                             3523.
Df Residuals:                     294   BIC:                             3545.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -751.2139   4456.871     -0.169      0.8